# Milestone 4 : fiabiliser la solution et analyser les gains

## 1. Suites de tests

Les tests portent sur le moteur du milestone 3, découpé en modules dans le package `backend/` (`couts`, `jours_fixes`, `charge`, `contraintes`, ...), dont `main.py` enchaîne les étapes. Le notebook appelle `main.executer()` : le moteur tourne une fois (planning final, sorties 1 et 2), sans rien afficher ni écrire. Les tests appellent directement les fonctions des modules (`couts.cout_transport`, `jours_fixes.consolider_couple`, ...) et les résultats du moteur (`r.planning_complet`, `r.sortie2`, ...).

Chaque test est une petite fonction avec des `assert`. Le décorateur `@test` l'exécute tout de suite et note son résultat : OK, ÉCHEC (une vérification est fausse, avec le message) ou ERREUR (le code a planté). Le bilan de tous les tests est dans la partie 1.5.

In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "..")
import main
from backend import charge, config, contraintes, couts, donnees, formatage, jours_fixes

r = main.executer()   # le moteur tourne une fois (planning final, sorties 1 et 2), sans rien écrire ni afficher

RESULTATS = {}   # (section, test) -> résultat ; relancer une cellule remplace ses résultats


def test(section, description):
    # Décorateur : exécute le test tout de suite et note son résultat
    def executer(fonction):
        try:
            fonction()
            resultat, detail = "OK", ""
        except AssertionError as e:
            resultat, detail = "ÉCHEC", str(e)
        except Exception as e:
            resultat, detail = "ERREUR", f"{type(e).__name__} : {e}"
        RESULTATS[(section, description)] = {"Section": section, "Test": description, "Résultat": resultat, "Détail": detail}
        return fonction
    return executer


def resultats(section):
    # Résultats d'une section, dans l'ordre des tests (le détail d'un échec est affiché dans la partie 1.5)
    table = pd.DataFrame([r for r in RESULTATS.values() if r["Section"] == section]).drop(columns=["Section", "Détail"])
    return table.style.hide(axis="index").map(lambda v: "color: #b00020; font-weight: bold" if v in ("ÉCHEC", "ERREUR") else "",
                                              subset=["Résultat"])


J = pd.Timestamp
P, L = r.planning_complet, r.lignes_complet   # planning final (une ligne par camion) et lignes de besoin livrées
print(f"Moteur chargé : {len(P)} camions ({int(P['reserve'].sum())} réservés), {len(L)} lignes de besoin, "
      f"du {donnees.DEBUT:%d/%m} au {donnees.FIN:%d/%m}")

### 1.1 Tests unitaires des fonctions

Les fonctions clés du moteur sont testées sur des cas construits à la main, dont on connaît le résultat : la grille tarifaire, la date de départ des camions, la consolidation d'un couple et le contrôle du lissage. Les dates sont celles de la semaine du lundi 16/11/2026.

In [ ]:
ROLES = {
    "tranche": "Trouve la tranche de la grille tarifaire (distance ou poids) à appliquer : la première borne supérieure ou égale à la valeur",
    "cout_transport": "Calcule le prix d'un camion : coût de la grille × (1 + marge de 239,23 %) × surcharge diesel de 1,1943",
    "date_camion": "Choisit le jour de départ du camion qui livre un besoin : le dernier jour fixe du couple avant ce besoin",
    "profil_utilise": "Vérifie qu'un couple à 2 ou 3 jours fixes livre au moins une semaine sur tous ses jours (C11)",
    "consolider_couple": "Regroupe les besoins d'un couple fournisseur × usine dans des camions, sans dépasser 24,5 t ni 110,4 m³ ni 7 jours d'avance",
    "hors_tolerance": "Contrôle du lissage : repère les semaines où une usine reçoit beaucoup plus de camions certains jours que d'autres",
}


def lignes_test(*lignes):
    # Lignes de besoin d'un couple fictif : (date du besoin, poids en kg, volume en m³)
    return pd.DataFrame([{"need_date": J(d), "weight": w, "volume": v} for d, w, v in lignes])


@test("1.1", "tranche : 30 km -> tranche 30 km ; 30,1 km -> tranche 50 km ; 651 km -> tranche > 650 km ; 4 000,5 kg -> tranche > 4 000 kg")
def _():
    assert couts.tranche(30, couts.TRANCHES_KM) == "30" and couts.tranche(30.1, couts.TRANCHES_KM) == "50"
    assert couts.tranche(651, couts.TRANCHES_KM) == ">650" and couts.tranche(4000.5, couts.TRANCHES_KG) == ">4000"


@test("1.1", "cout_transport : 300 km et 1 000 kg -> 335,34 €, le prix de l'exemple de la rate sheet")
def _():
    assert couts.cout_transport(1000, 300) == 335.34, couts.cout_transport(1000, 300)


@test("1.1", "date_camion : besoin le mercredi 18/11, livraison le lundi -> camion le lundi 16/11 ; besoin le vendredi 20/11, "
             "livraison lundi et jeudi -> camion le jeudi 19/11 ; besoin le lundi 09/11, livraison le jeudi -> camion le jeudi 05/11, "
             "dans la semaine ajoutée avant le planning")
def _():
    assert jours_fixes.date_camion(J("2026-11-18"), (0,)) == J("2026-11-16")
    assert jours_fixes.date_camion(J("2026-11-20"), (0, 3)) == J("2026-11-19")
    assert jours_fixes.date_camion(J("2026-11-09"), (3,)) == J("2026-11-05")


@test("1.1", "profil_utilise : profil lundi + jeudi, camions le lundi 16/11 et le jeudi 19/11 -> utilisé ; camions le lundi 16/11 "
             "et le jeudi 26/11 (jamais les deux la même semaine) -> pas utilisé ; profil d'un seul jour -> toujours utilisé")
def _():
    assert jours_fixes.profil_utilise([J("2026-11-16"), J("2026-11-19")], (0, 3))
    assert not jours_fixes.profil_utilise([J("2026-11-16"), J("2026-11-26")], (0, 3))
    assert jours_fixes.profil_utilise([J("2026-11-16"), J("2026-11-23")], (0,))


@test("1.1", "consolider_couple : 3 besoins de la semaine (1 t au total) -> 1 seul camion, le lundi ; 3 besoins de 10 t -> "
             "2 camions (30 t ne tiennent pas dans 24,5 t) ; 2 besoins de 60 m³ -> 2 camions (120 m³ > 110,4 m³)")
def _():
    semaine = jours_fixes.consolider_couple(lignes_test(("2026-11-16", 500, 2), ("2026-11-17", 300, 1), ("2026-11-20", 200, 1)), (0,))
    assert len(semaine) == 1 and semaine["date"].iloc[0] == J("2026-11-16")
    lourd = jours_fixes.consolider_couple(lignes_test(("2026-11-16", 10000, 1), ("2026-11-17", 10000, 1), ("2026-11-18", 10000, 1)), (0,))
    assert len(lourd) == 2 and (lourd["poids"] <= config.POIDS_MAX).all()
    volumineux = jours_fixes.consolider_couple(lignes_test(("2026-11-16", 100, 60), ("2026-11-17", 100, 60)), (0,))
    assert len(volumineux) == 2 and (volumineux["volume"] <= config.VOLUME_MAX).all()


@test("1.1", "consolider_couple : un besoin 7 jours après le camion du lundi part dans ce camion ; à 8 jours, "
             "il part dans le camion de la semaine suivante")
def _():
    assert len(jours_fixes.consolider_couple(lignes_test(("2026-11-16", 100, 1), ("2026-11-23", 100, 1)), (0,))) == 1
    assert len(jours_fixes.consolider_couple(lignes_test(("2026-11-16", 100, 1), ("2026-11-24", 100, 1)), (0,))) == 2


@test("1.1", "hors_tolerance : 2-2-2-2-2 camions du lundi au vendredi -> semaine équilibrée ; 10-0-0-0-0 -> hors tolérance ; "
             "3-2-2-2-1 -> équilibrée (écart d'un seul camion) ; 1 seul camion dans la semaine -> pas contrôlée")
def _():
    semaines = np.array([[2, 2, 2, 2, 2], [10, 0, 0, 0, 0], [3, 2, 2, 2, 1], [1, 0, 0, 0, 0]], dtype=float)
    assert charge.hors_tolerance(semaines).tolist() == [False, True, False, False]


# Tableau : la fonction, son rôle dans le moteur, le cas testé et le résultat
tests_11 = pd.DataFrame([r for r in RESULTATS.values() if r["Section"] == "1.1"])
tests_11[["Fonction", "Ce qu'on vérifie"]] = tests_11["Test"].str.split(" : ", n=1, expand=True)
tests_11["Rôle de la fonction"] = tests_11["Fonction"].map(ROLES)
(tests_11[["Fonction", "Rôle de la fonction", "Ce qu'on vérifie", "Résultat"]].style.hide(axis="index")
 .set_properties(**{"text-align": "left", "white-space": "normal"})
 .map(lambda v: "color: #b00020; font-weight: bold" if v in ("ÉCHEC", "ERREUR") else "", subset=["Résultat"]))

### 1.2 Tests du respect des contraintes

Les contraintes ont été définies au milestone 3 (partie 1 « Contraintes et objectif »). Chacune y a sa fonction de vérification, qui retourne les cas qui la violent (une table vide = contrainte respectée) :

| # | Contrainte | Fonction |
|---|---|---|
| C1 | Un camion ne dépasse pas 24,5 t | `c1_poids` |
| C2 | Un camion ne dépasse pas 110,4 m³ | `c2_volume` |
| C3 | Chaque ligne est livrée, une seule fois, entière | `c3_exhaustivite` |
| C4 | Un camion ne transporte qu'un seul couple | `c4_un_couple` |
| C5 | Jamais de retard | `c5_retard` |
| C6 | Au plus 7 jours d'avance | `c6_avance` |
| C7 | Livraison un jour ouvré, dans l'horizon du planning (semaine ajoutée avant le planning comprise) | `c7_jours_ouvres` |
| C8 | Jours fixes : 1 à 3 jours par couple, chaque camion part un de ces jours, sans exception | `c8_jours_fixes` |
| C9 | Au plus 1 camion par couple et par jour, sauf besoin d'un jour > 24,5 t | `c9_un_camion_par_jour` |
| C10 | Lissage : aucun jour ne s'écarte de plus de 35 % de la moyenne de la semaine | `c10_lissage` |
| C11 | Profil utilisé : un couple à 2 ou 3 jours fixes livre au moins une semaine sur tous ses jours fixes | `c11_profil_utilise` |

Au milestone 3 (partie 5), ces fonctions ont été appliquées au planning final : toutes les contraintes dures sont respectées et 2 semaines sur 20 restent hors tolérance pour le lissage.

Ici, on teste deux choses :
1. le planning final respecte toujours les contraintes : on rejoue les fonctions du milestone 3 sur le planning produit par `main.py` 

2. les fonctions de vérification détectent vraiment une erreur : une fonction qui renverrait toujours une table vide passerait le premier test sans rien vérifier. Pour chaque contrainte, on introduit donc volontairement une erreur dans une copie du planning (un camion de 24,5 t + 1 kg, une ligne livrée un jour en retard, un camion un samedi, ...) et on vérifie que la fonction la signale.

In [ ]:
FONCTIONS = {code_c: fonction for code_c, _, _, fonction in contraintes.CONTRAINTES}


def violations(code_c, planning, lignes):
    # Violations d'une contrainte (pour C9 : seulement les cas non justifiés par un besoin d'un jour > 24,5 t)
    args = (planning, lignes, donnees.IDS_ATTENDUS) if code_c == "C3" else (planning, lignes)
    v = FONCTIONS[code_c](*args)
    return v[~v["justifie"]] if code_c == "C9" else v


# ---------- Erreurs introduites volontairement, une par contrainte ----------
opti = P[~P["reserve"]]
i0 = opti.index[0]                                                          # un camion optimisé quelconque
multi = L.groupby("camion_id").size().loc[lambda n: n > 1].index[0]         # un camion qui transporte plusieurs lignes
couples_c9 = set(map(tuple, contraintes.c9_un_camion_par_jour(P, L)[["plant", "supplier_id"]].values))
i9 = next(i for i, r in opti.iterrows() if (r["plant"], r["supplier_id"]) not in couples_c9)
i8 = i0
i11 = next(i for i, prof in opti["profil"].items() if len(prof) == 1)        # un camion d'un couple à 1 jour fixe


def avec_erreur(code_c):
    # Copie du planning et des lignes avec une erreur qui viole la contrainte
    p, l = P.copy(), L.copy()
    if code_c == "C1":
        p.loc[i0, "poids"] = config.POIDS_MAX + 1                                    # camion de 24,5 t + 1 kg
    elif code_c == "C2":
        p.loc[i0, "volume"] = config.VOLUME_MAX + 1                                  # camion de 110,4 m³ + 1 m³
    elif code_c == "C3":
        l = l.drop(index=l.index[0])                                            # ligne oubliée
    elif code_c == "C4":
        l.loc[l.index[l["camion_id"] == multi][0], "supplier_id"] = "SUP_99999"  # ligne d'un autre fournisseur
    elif code_c == "C5":
        l.loc[l.index[0], "date_livraison"] = l.loc[l.index[0], "need_date"] + pd.Timedelta(days=1)   # 1 jour de retard
    elif code_c == "C6":
        l.loc[l.index[0], "date_livraison"] = l.loc[l.index[0], "need_date"] - pd.Timedelta(days=8)   # 8 jours d'avance
    elif code_c == "C7":
        p.loc[i0, "date"] = J("2026-11-21")                                     # camion un samedi
    elif code_c == "C8":
        date, profil = p.loc[i8, "date"], p.loc[i8, "profil"]                   # camion hors de ses jours fixes
        p.loc[i8, "date"] = date + pd.Timedelta(days=next(j for j in range(5) if j not in profil) - date.day_of_week)
    elif code_c == "C9":
        p = pd.concat([p, p.loc[[i9]]], ignore_index=True)                      # 2 camions le même jour sans raison
    elif code_c == "C11":
        plant, supplier, jour = p.loc[i11, "plant"], p.loc[i11, "supplier_id"], p.loc[i11, "date"].day_of_week
        du_couple = (p["plant"] == plant) & (p["supplier_id"] == supplier)       # 2e jour fixe jamais utilisé
        p["profil"] = [tuple(sorted({jour, (jour + 1) % 5})) if m else prof for m, prof in zip(du_couple, p["profil"])]
    elif code_c == "C10":
        renfort = p[p["plant"] == donnees.USINES[0]].head(1).assign(date=J("2026-11-16"))
        p = pd.concat([p] + [renfort] * 15, ignore_index=True)                  # 15 camions de plus un lundi
    return p, l


@test("1.2", "Le planning final respecte les contraintes dures C1 à C9 et C11 ; C10 : au plus 2 semaines hors tolérance")
def _():
    nb = {c: len(violations(c, P, L)) for c, _, _, _ in contraintes.CONTRAINTES}
    violees = {c: n for c, n in nb.items() if c != "C10" and n > 0}
    assert not violees, f"contraintes violées : {violees}"
    assert nb["C10"] <= 2, f"{nb['C10']} semaines hors tolérance"


@test("1.2", "Chaque fonction de vérification (C1 à C11) détecte l'erreur qu'on lui introduit")
def _():
    ratees = [c for c, _, _, _ in contraintes.CONTRAINTES if len(violations(c, *avec_erreur(c))) <= len(violations(c, P, L))]
    assert not ratees, f"erreur non détectée par : {ratees}"


resultats("1.2")

### 1.3 Cas limites et hypothèses de données manquantes

Les hypothèses prises aux milestones 1 à 3 pour les données incomplètes et les cas particuliers sont vérifiées sur le planning final :
- ligne de plus de 24,5 t : exclue du moteur, en attendant la réponse du client ;
- poids manquant : la ligne ne peut pas être consolidée, elle a son propre camion réservé, le jour du besoin, chiffré au tarif > 4 000 kg 

- distance manquante : le camion est chiffré au tarif > 650 km, et la distance est affichée « inconnue » dans la sortie 1 

- début du planning : on peut livrer dans la semaine qui précède le planning (S45, à partir du lundi 02/11). Les premiers besoins partent donc, comme les autres, le dernier jour fixe qui les précède, au plus 7 jours avant.

- cas impossibles : une ligne sans poids garde toujours son camion réservé, le jour du besoin (règle absolue), même si le couple reçoit alors 2 camions ce jour-là ou si ce n'est pas un de ses jours fixes ; de même, un couple dont le besoin d'un jour dépasse 24,5 t reçoit 2 camions. Ces cas ne sont pas cachés : chacun est notifié dans les alertes (`data/gold/alertes.csv`), avec sa cause et l'action à mener.

In [ ]:
q = donnees.quarantaine


@test("1.3", "Ligne de plus de 24,5 t : exclue du moteur et absente des sorties")
def _():
    lourdes = q[q["weight"] > config.POIDS_MAX]
    assert len(lourdes) == 1, f"{len(lourdes)} ligne(s) de plus de 24,5 t"
    cles = ["need_date", "supplier_id", "material", "plant", "unloading_point"]
    assert lourdes[cles].merge(L[cles]).empty, "la ligne lourde est livrée"


@test("1.3", "Poids manquant : une ligne = un camion réservé, le jour du besoin, tarif > 4 000 kg")
def _():
    reserves, lignes_r = P[P["reserve"]], L[L["reserve"]]
    assert len(reserves) == q["weight"].isna().sum() == len(lignes_r)
    assert (lignes_r["date_livraison"] == lignes_r["need_date"]).all()
    assert np.allclose(reserves["cout"], [couts.cout_transport(couts.POIDS_PAR_DEFAUT, couts.distances[s]) for s in reserves["supplier_id"]])


@test("1.3", "Distance manquante : camions chiffrés au tarif > 650 km, distance « inconnue » dans la sortie 1")
def _():
    camions = P[P["supplier_id"].isin(couts.SANS_DISTANCE)]
    assert len(camions) > 0, "aucun fournisseur sans distance dans le planning"
    poids = camions["poids"].fillna(couts.POIDS_PAR_DEFAUT)
    assert np.allclose(camions["cout"], [couts.cout_transport(w, couts.DISTANCE_PAR_DEFAUT) for w in poids])
    assert r.sortie1.loc[r.sortie1["fournisseur"].isin(couts.SANS_DISTANCE), "distance_km"].isna().all()


@test("1.3", "Début du planning : tous les camions partent un jour fixe ; ceux d'avant le 09/11 partent dans la semaine ajoutée "
             "(à partir du lundi 02/11), au plus 7 jours avant le besoin, et ne livrent que des besoins de la première semaine")
def _():
    opti = P[~P["reserve"]]
    hors = [d.day_of_week not in prof for d, prof in zip(opti["date"], opti["profil"])]
    assert not any(hors), f"{sum(hors)} camion(s) hors de leurs jours fixes"
    avant = opti[opti["date"] < donnees.DEBUT]
    assert len(avant) > 0 and (avant["date"] >= donnees.DEBUT_LIVRAISONS).all()
    lignes_avant = L[L["camion_id"].isin(avant["camion_id"])]
    assert (lignes_avant["need_date"] < donnees.DEBUT + pd.Timedelta(days=7)).all(), "besoin d'après la première semaine"
    assert lignes_avant["avance_jours"].between(0, 7).all()


@test("1.3", "Cas impossibles notifiés : chaque jour où un couple reçoit 2 camions ou plus, et chaque camion réservé hors des "
             "jours fixes du couple, figure dans les alertes avec sa cause et l'action à mener")
def _():
    a = r.alertes
    c9 = contraintes.c9_un_camion_par_jour(P, L)
    assert c9["justifie"].all(), "2 camions un même jour sans cause"
    a9 = a[a["regle"].str.startswith("C9")]
    assert set(zip(c9["plant"], c9["supplier_id"], c9["date"])) == set(zip(a9["usine"], a9["fournisseur"], a9["date"]))
    profils = P[~P["reserve"]].groupby(["plant", "supplier_id"])["profil"].first()
    reserves = P[P["reserve"]]
    hors = sum(profils.get((p, s)) is None or d.day_of_week not in profils.get((p, s))
               for p, s, d in zip(reserves["plant"], reserves["supplier_id"], reserves["date"]))
    assert (a["regle"].str.startswith("C8")).sum() == hors, "camion réservé hors jours fixes non notifié"
    assert a["cause"].str.len().gt(0).all() and a["a_faire"].str.len().gt(0).all()


resultats("1.3")

### 1.4 Exhaustivité des matériaux livrés et anticipation des besoins

le moteur réorganise les livraisons (il regroupe des besoins de plusieurs jours dans un même camion), mais il ne doit rien perdre, rien inventer et ne jamais mettre une usine en rupture. Concrètement :
- exhaustivité : chaque ligne de besoin du planning d'entrée (fichiers silver et quarantaine) se retrouve dans un camion, une seule fois ; une ligne oubliée serait une pièce jamais livrée, une ligne en double un camion chargé pour rien 

- matériaux : pour chaque material, on compare la quantité demandée par l'usine (planning d'entrée) à la quantité chargée dans les camions (sortie 2). Le regroupement change les jours et les camions, mais pas les quantités : les deux doivent être égales 

- anticipation : chaque ligne arrive au plus tard le jour de son besoin (pas de rupture à l'usine), et au plus 7 jours avant (l'usine n'a pas à stocker plus d'une semaine).

Le graphique montre avec combien de jours d'avance chaque ligne est livrée : une barre par nombre de jours, de « le jour même » à « 7 jours avant », la limite autorisée. Un camion part un des jours fixes du couple et emporte les besoins des jours suivants : plus le besoin est loin du jour fixe, plus il est livré tôt.

In [ ]:
# ---------- Ce qui est comparé ----------
CLES = ["plant", "supplier_id", "material", "unloading_point"]
entree = pd.concat([donnees.besoins, donnees.lignes_reserves])                          # planning d'entrée : besoins à livrer
demande = entree.groupby(CLES)["schedule_quantity"].sum()                   # quantité demandée par material
livre = (r.sortie2.rename(columns={"usine": "plant", "fournisseur": "supplier_id", "quai": "unloading_point"})
         .groupby(CLES)["quantite"].sum())                                  # quantité chargée dans les camions (sortie 2)


@test("1.4", f"Aucune ligne de besoin oubliée ni livrée deux fois : les {len(donnees.IDS_ATTENDUS)} lignes du planning d'entrée "
             f"sont chacune dans un camion, une seule fois, et la sortie 2 les compte toutes")
def _():
    assert L["id_ligne"].is_unique and set(L["id_ligne"]) == donnees.IDS_ATTENDUS and L["camion_id"].notna().all()
    assert r.sortie2["nb_lignes"].sum() == len(donnees.IDS_ATTENDUS) and set(r.sortie2["camion_id"]) == set(P["camion_id"])


@test("1.4", f"Chaque material est livré en quantité exacte : pour les {len(demande)} combinaisons material × quai × couple, "
             f"quantité demandée (planning d'entrée) = quantité chargée dans les camions (sortie 2), au total "
             f"{formatage.nombre(demande.sum())} pièces des deux côtés")
def _():
    ecarts = demande.sub(livre, fill_value=0).abs()
    assert (ecarts < 1e-6).all(), f"{(ecarts >= 1e-6).sum()} materials avec un écart"


@test("1.4", f"Aucune usine en rupture, aucun stock de plus d'une semaine : pour les {len(L)} lignes, date de livraison "
             f"entre 7 jours avant et le jour du besoin (avance de {L['avance_jours'].min()} à {L['avance_jours'].max()} jours)")
def _():
    assert L["avance_jours"].between(0, 7).all(), L["avance_jours"].describe().to_string()


tests_14 = pd.DataFrame([r for r in RESULTATS.values() if r["Section"] == "1.4"])
tests_14[["Ce qu'on veut garantir", "Comment on le vérifie"]] = tests_14["Test"].str.split(" : ", n=1, expand=True)
display(tests_14[["Ce qu'on veut garantir", "Comment on le vérifie", "Résultat"]].style.hide(axis="index")
        .set_properties(**{"text-align": "left", "white-space": "normal"})
        .map(lambda v: "color: #b00020; font-weight: bold" if v in ("ÉCHEC", "ERREUR") else "", subset=["Résultat"]))

# ---------- Avec combien de jours d'avance les usines reçoivent-elles leurs besoins ? ----------
import matplotlib.pyplot as plt

avance = L["avance_jours"].value_counts().sort_index().reindex(range(8), fill_value=0)
part = avance / avance.sum()
fig, ax = plt.subplots(figsize=(9, 4.5))
barres = ax.bar(avance.index, part * 100, color="royalblue", edgecolor="black")
ax.bar_label(barres, labels=[f"{formatage.pourcent(p)}\n({n} lignes)" for p, n in zip(part, avance)], fontsize=8, padding=2)
ax.set_xticks(range(8), ["le jour\nmême"] + [f"{j} j avant" for j in range(1, 8)])
ax.set_xlabel("Date de livraison par rapport à la date du besoin")
ax.set_ylabel("Part des lignes de besoin (%)")
ax.set_title(f"Avance des livraisons sur le besoin ({len(L)} lignes) : jamais de retard, jamais plus de 7 jours")
ax.margins(y=0.2)
ax.grid(axis="y", alpha=0.3)
plt.show()

### 1.5 Résultats des tests

In [ ]:
bilan_tests = pd.DataFrame(RESULTATS.values())
synthese = (bilan_tests.groupby("Section", sort=False)["Résultat"].value_counts().unstack(fill_value=0)
            .reindex(columns=["OK", "ÉCHEC", "ERREUR"], fill_value=0))
synthese.insert(0, "Tests", synthese.sum(axis=1))
synthese.index = synthese.index.map({"1.1": "1.1 Tests unitaires", "1.2": "1.2 Respect des contraintes",
                                     "1.3": "1.3 Cas limites", "1.4": "1.4 Exhaustivité et anticipation"})
synthese.loc["Total"] = synthese.sum()
display(synthese)

echecs = bilan_tests[bilan_tests["Résultat"] != "OK"]
print("Tous les tests passent." if echecs.empty else f"{len(echecs)} test(s) en échec :")
echecs if len(echecs) else None

Résultat : les 17 tests passent.
- Les fonctions clés du moteur donnent les résultats attendus sur des cas construits à la main, y compris aux bornes : tranche exacte, 7 jours d'avance acceptés et 8 refusés, capacité atteinte en poids ou en volume, profil à 2 jours utilisé ou non. Le prix calculé retrouve l'exemple de la rate sheet (335,34 €).

- Le planning final respecte toutes les contraintes dures définies au milestone 3, et le lissage reste à 2 semaines hors tolérance sur 20. Chaque fonction de vérification détecte l'erreur qu'on lui introduit : les contrôles du milestone 3 sont donc fiables.

- Les hypothèses sur les données manquantes sont appliquées comme prévu : la ligne de plus de 24,5 t est exclue, chaque ligne sans poids a son camion réservé, et les fournisseurs sans distance sont chiffrés au tarif le plus haut. Aucun camion ne part hors des jours fixes : les premiers besoins partent dans la semaine ajoutée avant le planning (à partir du 02/11). Les cas impossibles (lignes sans poids, besoin d'un jour > 24,5 t) sont tous notifiés dans les alertes.

- Rien n'est perdu ni créé : chaque ligne est livrée une fois, et la quantité livrée est égale à la quantité demandée pour chaque material, quai et couple. Aucune usine n'est livrée en retard. 13 % des lignes arrivent le jour de leur besoin et 16 % avec 7 jours d'avance, la limite autorisée : c'est le prix du regroupement sur un seul jour fixe par semaine pour presque tous les couples.

### Nouvelles distributions (taux de remplissage, coûts, ...)

Cette partie compare la situation actuelle (baseline du milestone 2) au planning consolidé produit par le moteur du milestone 3. 
Les deux sont chargés depuis leurs fichiers. 
Les mêmes indicateurs sont calculés des deux côtés, avec la même fonction (`preparer`) : coût avec la grille tarifaire du milestone 2, taux de remplissage, semaine et jour.

Les **134 camions réservés** (lignes sans poids) sont identiques avant et après. Ils sont **exclus des distributions** de remplissage, car on ne connaît pas leur chargement, mais **inclus dans les coûts totaux**.

Dans un premier temps nous allons récuperer tous les fichiers nécessaire, soit la baseline et notre optimisation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#  Fichiers 
BASELINE_PATH = "../data/baseline/baseline_camion.csv"     # avant : une ligne par camion (milestone 2)
SORTIE2_PATH = "../data/gold/sortie2_camions.csv"           # après : une ligne par camion × material × quai (milestone 3)
SILVER_CLEAN_PATH = "../data/silver/released_schedule_cleaned.csv"
QUARANTAINE_PATH = "../data/silver/released_schedule_quarantine.csv"
RATE_SHEET_PATH = "../data/bronze/rate_sheet.csv"
DISTANCES_PATH = "../data/bronze/distances.csv"

# Paramètres (mêmes valeurs qu'aux milestones 2 et 3)
POIDS_MAX = 24500     # kg
VOLUME_MAX = 110.4    # m³
FREIGHT_VALUE_MARGIN = 2.3923   # +239,23 %
DIESEL_SURCHARGE = 1.1943

COULEURS = {"Avant": "#b8c4cc", "Après": "#3f7489"}   # gris = baseline, bleu = planning consolidé

#  Grille tarifaire 
couts = pd.read_csv(RATE_SHEET_PATH, index_col="km_max")
TRANCHES_KM = [int(km) for km in couts.index if not km.startswith(">")]
TRANCHES_KG = [int(kg) for kg in couts.columns if not kg.startswith(">")]
POIDS_PAR_DEFAUT = TRANCHES_KG[-1] + 1      # poids inconnu -> tranche > 4 000 kg
DISTANCE_PAR_DEFAUT = TRANCHES_KM[-1] + 1   # distance inconnue -> tranche > 650 km


def tranche(valeur, tranches):
    # Première tranche >= valeur (on majore), sinon la tranche '>max'
    for borne in tranches:
        if valeur <= borne:
            return str(borne)
    return f">{tranches[-1]}"


def cout_transport(poids, distance):
    # Prix d'un camion en € = coût de la grille × (1 + marge) × surcharge diesel
    cout_base = couts.loc[tranche(distance, TRANCHES_KM), tranche(poids, TRANCHES_KG)]
    return float(round(cout_base * (1 + FREIGHT_VALUE_MARGIN) * DIESEL_SURCHARGE, 2))


# Mise en forme 
def nombre(x, decimales=1):
    return f"{x:,.{decimales}f}".replace(",", " ").replace(".", ",").removesuffix(",0")
def pourcent(x, decimales=1):
    return f"{x:.{decimales}%}".replace(".", ",").replace("%", " %")
def eur(x):
    return f"{x:,.0f} €".replace(",", " ")


#  Avant : baseline du milestone 2 
distances = pd.read_csv(DISTANCES_PATH, encoding="utf-8-sig").set_index("supplier_id")["distance"]
avant = pd.read_csv(BASELINE_PATH, parse_dates=["date"])[["camion_id", "supplier_id", "plant", "date", "poids", "volume", "reserve"]]

#  Après : planning consolidé du milestone 3, reconstruit camion par camion depuis la sortie 2 
sortie2 = pd.read_csv(SORTIE2_PATH, parse_dates=["date_livraison"])

# La sortie 2 n'a pas le volume : on le reprend dans les lignes de besoin (silver + quarantaine).
# Une ligne de la sortie 2 peut regrouper plusieurs dates de besoin ("2026-11-16, 2026-11-17") : on la découpe par date.
besoins = pd.concat([pd.read_csv(SILVER_CLEAN_PATH, parse_dates=["need_date"]),
                     pd.read_csv(QUARANTAINE_PATH, parse_dates=["need_date"])], ignore_index=True)
CLE = ["need_date", "supplier_id", "plant", "material", "unloading_point"]
volumes = besoins.groupby(CLE)["volume"].sum(min_count=1)

detail = (sortie2.rename(columns={"fournisseur": "supplier_id", "usine": "plant", "quai": "unloading_point"})
                 .assign(need_date=lambda d: d["besoins"].str.split(", "))
                 .explode("need_date"))
detail["need_date"] = pd.to_datetime(detail["need_date"])
detail["volume"] = volumes.reindex(pd.MultiIndex.from_frame(detail[CLE])).values

apres = (detail.groupby("camion_id")
               .agg(supplier_id=("supplier_id", "first"), plant=("plant", "first"), date=("date_livraison", "first"),
                    volume=("volume", lambda v: v.sum(min_count=1)), reserve=("reserve", "first"))
               .reset_index())
apres["poids"] = sortie2.groupby("camion_id")["poids_kg"].sum(min_count=1).loc[apres["camion_id"]].values  # NaN = réservé


#  Indicateurs communs avant/après
def preparer(camions, version):
    c = camions.copy()
    c["version"] = version
    c["distance"] = c["supplier_id"].map(distances).fillna(DISTANCE_PAR_DEFAUT)   # sans distance -> > 650 km
    c["cout"] = [cout_transport(POIDS_PAR_DEFAUT if r else p, d)                   # réservé -> > 4 000 kg
                 for p, d, r in zip(c["poids"], c["distance"], c["reserve"])]
    c["remplissage_poids"] = c["poids"] / POIDS_MAX
    c["remplissage_volume"] = c["volume"] / VOLUME_MAX
    c["semaine"] = c["date"].dt.isocalendar().week
    c["jour"] = c["date"].dt.day_of_week          # 0 = lundi
    return c


camions = pd.concat([preparer(avant, "Avant"), preparer(apres, "Après")], ignore_index=True)
connus = camions[~camions["reserve"]]   # camions dont le poids est connu, les seuls pour analyser le remplissage

#  Contrôle pour etre sur qu'on retrouve bien les chiffres des milestones 2 et 3 
controle = camions.groupby("version", sort=False).agg(camions=("camion_id", "size"), dont_reserves=("reserve", "sum"),
                                                      poids_total_t=("poids", "sum"), cout=("cout", "sum"))
controle["poids_total_t"] = (controle["poids_total_t"] / 1000).round(0)   # tonnes : l'écart d'arrondi (< 1 kg) disparaît
controle

Les 134 camions réservés sont exclus : on ne connaît pas leur poids. Il reste les camions dont le poids est connu, 1 607 avant (1 741 − 134) et 602 après (736 − 134). Comme leur nombre est très différent, on affiche la **part** des camions de chaque version dans chaque classe, et non leur nombre.

### Taux de remplissage par camion 

#### Poids et taux de remplissage des camions

Le taux de remplissage d'un camion, c'est son poids divisé par 24,5 t. Les classes suivent les tranches de poids de la grille tarifaire, car le milestone 2 a montré que le seuil de 4 000 kg est décisif : au-delà, le prix d'un camion ne bouge presque plus. Sous chaque classe, on affiche le taux de remplissage correspondant.

Les 134 camions réservés sont exclus : on ne connaît pas leur poids. Il reste les camions dont le poids est connu, 1 607 avant (1 741 − 134) et 602 après (736 − 134). Comme leur nombre est très différent, on affiche la **part** des camions de chaque version dans chaque classe, et non leur nombre.

In [ ]:
# ---------- Poids et taux de remplissage par camion, selon les tranches de la grille tarifaire ----------
# Les bornes sont toutes des tranches de poids de la grille : dans une classe, le prix d'un camion change peu
BORNES_KG = [0, 100, 300, 500, 1000, 2000, 4000, POIDS_MAX]
# Chaque classe affiche aussi son taux de remplissage (poids / 24,5 t)
LIBELLES_KG = [f"{a}-{b} kg\n({pourcent(BORNES_KG[i] / POIDS_MAX)}-{pourcent(BORNES_KG[i + 1] / POIDS_MAX)})"
               for i, (a, b) in enumerate(zip(["0", "100", "300", "500", "1 000", "2 000", "4 000"],
                                              ["100", "300", "500", "1 000", "2 000", "4 000", "24 500"]))]

effectifs = connus.groupby("version")["camion_id"].size()
classes_kg = (pd.crosstab(pd.cut(connus["poids"], bins=BORNES_KG, labels=LIBELLES_KG, include_lowest=True),
                          connus["version"], normalize="columns")     # part des camions de chaque version
              [["Avant", "Après"]])
mediane_kg = connus.groupby("version")["poids"].median()

ax = classes_kg.mul(100).plot(kind="bar", color=[COULEURS[v] for v in classes_kg.columns], edgecolor="black",
                              figsize=(11, 5.2), rot=0, width=0.8)
for conteneur in ax.containers:
    ax.bar_label(conteneur, fmt="%.0f %%", fontsize=8, padding=2)
ax.axvline(len(LIBELLES_KG) - 1.5, color="red", linestyle="--")      # entre « 2 000-4 000 » et « > 4 000 »
ax.annotate("fin de la grille tarifaire :\nau-delà, prix quasi fixe", xy=(len(LIBELLES_KG) - 1.45, 22),
            color="red", fontsize=9)
ax.set_title(f"Poids et taux de remplissage des camions, selon les tranches de la grille tarifaire\n"
             f"médiane : {nombre(mediane_kg['Avant'], 0)} kg ({pourcent(mediane_kg['Avant'] / POIDS_MAX)}) avant → "
             f"{nombre(mediane_kg['Après'], 0)} kg ({pourcent(mediane_kg['Après'] / POIDS_MAX)}) après "
             f"(hors camions réservés)")
ax.set_xlabel("Poids du camion (taux de remplissage = poids / 24,5 t)")
ax.set_ylabel("Part des camions (%)")
ax.legend(ax.containers, [f"Avant : baseline ({effectifs['Avant']} camions)",
                          f"Après : planning consolidé ({effectifs['Après']} camions)"], loc="upper center")
ax.grid(axis="y", alpha=0.5)
ax.set_axisbelow(True)
ax.margins(y=0.12)
plt.tight_layout()
plt.show()

# ---------- Chiffres pour l'analyse ----------
for version in ["Avant", "Après"]:
    p = connus.loc[connus["version"] == version, "poids"]
    print(f"{version} : médiane {nombre(p.median(), 0)} kg ({pourcent(p.median() / POIDS_MAX)}), "
          f"moyenne {nombre(p.mean(), 0)} kg | ≤ 500 kg : {pourcent((p <= 500).mean())} | "
          f"> 4 000 kg : {(p > 4000).sum()} camions ({pourcent((p > 4000).mean())}), "
          f"soit {pourcent(p[p > 4000].sum() / p.sum())} du poids transporté")

# Pourquoi le remplissage reste bas : le poids qu'un couple a à livrer en une semaine
poids_semaine = connus[connus["version"] == "Après"].groupby(["supplier_id", "plant", "semaine"])["poids"].sum()
print(f"Poids médian à livrer par couple et par semaine : {nombre(poids_semaine.median(), 0)} kg ; "
      f"{pourcent((poids_semaine <= 4000).mean())} des couples-semaines sous 4 000 kg")

#### Analyse : poids et taux de remplissage

**La distribution se décale vers la droite.** Le poids médian d'un camion passe de **323 kg à 806 kg**, soit un taux de remplissage médian de **1,3 % à 3,3 %** (×2,5). Les petits envois reculent nettement : 64 % des camions pesaient moins de 500 kg (2 % de remplissage) dans la baseline, contre 39 % après consolidation. La classe 100-300 kg, la plus fréquente avant (30 % des camions), tombe à 18 %.

**Le même poids part dans 2,7 fois moins de camions.** Les 1 588 t à livrer sont les mêmes avant et après (aucune ligne perdue, voir la partie 1.4). Elles partent dans 602 camions au lieu de 1 607. Le poids moyen d'un camion passe donc de 988 kg à 2 639 kg.

**La part des camions au-delà de 4 000 kg est multipliée par 3,7 : de 4,4 % à 16,3 %.** C'est le chiffre qui compte pour le prix : au-delà de 4 000 kg, le prix d'un camion est quasi fixe. Ces camions transportent maintenant **70 % du poids total**, contre 47 % avant : la majorité des marchandises voyage dans la tranche de prix la plus avantageuse au kilo.

**Le remplissage reste pourtant faible**, car les couples ont peu à livrer : en médiane, un couple n'a que 771 kg de besoins par semaine, et 85 % des couples-semaines restent sous 4 000 kg. Même regroupés dans un seul camion par semaine, ces besoins ne peuvent pas remplir un camion de 24,5 t. Pour aller plus loin, il faudrait lever deux contraintes : mélanger plusieurs fournisseurs dans un même camion (C4) ou livrer plus de 7 jours en avance (C6).

#### Coût de transport par tranche de poids

Le graphe précédent compte des camions ; celui-ci additionne leur coût, avec les mêmes tranches de poids. Sous chaque tranche, on affiche le coût moyen à la tonne transportée : c'est lui qui montre l'effet de la grille tarifaire dégressive. Les camions réservés sont exclus (leur coût est identique avant et après : 33 506 €).

In [ ]:
#  Où part l'argent : coût total par tranche de poids, avant / après 
classe_cout = pd.cut(connus["poids"], bins=BORNES_KG, labels=LIBELLES_KG, include_lowest=True)
cout_classe = connus.groupby([classe_cout, "version"], observed=False)["cout"].sum().unstack()[["Avant", "Après"]]
cout_tonne = connus.groupby("version").apply(lambda c: c["cout"].sum() / c["poids"].sum() * 1000)   # € par tonne
cout_tonne_classe = connus.groupby(classe_cout, observed=False).apply(lambda c: c["cout"].sum() / c["poids"].sum() * 1000)

ax = cout_classe.plot(kind="bar", color=[COULEURS[v] for v in cout_classe.columns], edgecolor="black",
                      figsize=(11, 5.2), rot=0, width=0.8)
for conteneur in ax.containers:
    ax.bar_label(conteneur, labels=[eur(v) for v in conteneur.datavalues], fontsize=8, padding=2)
ax.axvline(len(LIBELLES_KG) - 1.5, color="red", linestyle="--")
ax.set_xticks(range(len(LIBELLES_KG)),
              [f"{l.split(chr(10))[0]}\n{nombre(t, 0)} €/t" for l, t in zip(LIBELLES_KG, cout_tonne_classe)])
total = connus.groupby("version")["cout"].sum()
ax.set_title(f"Coût de transport par tranche de poids des camions (hors camions réservés)\n"
             f"total : {eur(total['Avant'])} avant → {eur(total['Après'])} après ; "
             f"coût à la tonne : {nombre(cout_tonne['Avant'], 0)} €/t → {nombre(cout_tonne['Après'], 0)} €/t")
ax.set_xlabel("Poids du camion (sous chaque tranche : coût moyen à la tonne transportée)")
ax.set_ylabel("Coût total sur la période (€)")
ax.legend(ax.containers, ["Avant : baseline", "Après : planning consolidé"], loc="upper center", ncol=2)
ax.grid(axis="y", alpha=0.5)
ax.set_axisbelow(True)
ax.set_ylim(0, cout_classe.values.max() * 1.25)
ax.yaxis.set_major_formatter(lambda v, _: nombre(v, 0))
plt.tight_layout()
plt.show()

# Chiffres pour l'analyse
for version in ["Avant", "Après"]:
    c = connus[connus["version"] == version]
    petits = c["poids"] <= 500
    print(f"{version} : coût total {eur(c['cout'].sum())}, coût médian d'un camion {nombre(c['cout'].median(), 0)} €, "
          f"coût à la tonne {nombre(cout_tonne[version], 0)} €/t | camions ≤ 500 kg : {pourcent(petits.mean())} des camions, "
          f"{pourcent(c.loc[petits, 'poids'].sum() / c['poids'].sum())} du poids, "
          f"{pourcent(c.loc[petits, 'cout'].sum() / c['cout'].sum())} du coût ({eur(c.loc[petits, 'cout'].sum())})")

#### Analyse : coût par tranche de poids

**La grille tarifaire rend les petits camions très chers à la tonne.** Une tonne transportée dans un camion de moins de 100 kg coûte en moyenne 775 €, contre 22 € dans un camion de plus de 4 000 kg : 35 fois moins. L'économie vient donc du fait que le poids est déplacé vers les tranches les moins chères.

**C'est ce que fait la consolidation.** Dans la baseline, les camions de moins de 500 kg représentaient 64 % des camions et seulement 13 % du poids transporté, mais **39 % du coût** (55 807 €). Après consolidation, ils ne pèsent plus que 17 % du coût (13 488 €). Toutes les tranches sous 2 000 kg baissent fortement.

**Seule la tranche au-delà de 4 000 kg coûte plus cher après** (17 182 € → 24 737 €). C'est voulu : elle compte plus de camions, mais c'est la tranche où la tonne coûte le moins cher. Elle transporte désormais 70 % du poids pour 31 % du coût.

**Au total, hors camions réservés, le coût passe de 143 815 € à 78 689 € (−45 %), et le coût à la tonne de 91 €/t à 50 €/t.** Chaque camion coûte un peu plus cher (médiane de 71 € à 104 €, car il est plus lourd), mais il y en a 2,7 fois moins.

#### Camions reçus par jour et par usine

Ce graphe reprend la charge des usines du milestone 2, jour par jour et usine par usine. Les camions réservés sont comptés : l'usine les reçoit à ses quais comme les autres. Le milestone 3 comparait le planning avant et après lissage. Ici, on le compare à la situation actuelle (baseline).

In [ ]:
#  Camions reçus chaque jour ouvré, par usine : avant / après 
# Tous les camions sont comptés (camions réservés compris) 
par_jour = (camions.groupby(["plant", "date", "version"]).size().unstack(fill_value=0)
                   .reindex(columns=["Avant", "Après"]))
dates = sorted(camions["date"].unique())
lundis = [d for d in dates if pd.Timestamp(d).day_of_week == 0]

fig, axes = plt.subplots(2, 2, figsize=(14, 7.5), sharex=True)
for ax, plant in zip(axes.flat, sorted(camions["plant"].unique())):
    serie = par_jour.loc[plant].reindex(dates, fill_value=0)
    x = np.arange(len(dates))
    ax.bar(x, serie["Avant"], color=COULEURS["Avant"], edgecolor="black", linewidth=0.4, label="Avant : baseline")
    ax.bar(x, serie["Après"], width=0.45, color=COULEURS["Après"], label="Après : planning consolidé")
    for i, d in enumerate(dates):                         # un trait au début de chaque semaine
        if pd.Timestamp(d).day_of_week == 0:
            ax.axvline(i - 0.5, color="grey", linewidth=0.6, linestyle=":")
    ax.set_title(f"{plant} : jour le plus chargé {serie['Avant'].max()} → {serie['Après'].max()} camions")
    ax.set_xticks([dates.index(l) for l in lundis], [pd.Timestamp(l).strftime("%d/%m") for l in lundis])
    ax.grid(axis="y", alpha=0.5)
    ax.set_axisbelow(True)
for ax in axes[:, 0]:
    ax.set_ylabel("Camions reçus")
for ax in axes[1]:
    ax.set_xlabel("Jours ouvrés (étiquette = lundi de chaque semaine)")
axes[0, 0].legend(loc="upper left")
fig.suptitle("Camions reçus par jour ouvré et par usine (camions réservés compris)", fontsize=14)
plt.tight_layout()
plt.show()

#  Chiffres pour l'analyse 
stats = par_jour.reset_index().melt(id_vars=["plant", "date"], var_name="version", value_name="nb")
print(stats.groupby(["plant", "version"])["nb"].agg(["max", "mean", "std"]).round(1).unstack())
total_jour = camions.groupby(["date", "version"]).size().unstack()[["Avant", "Après"]]
print(f"Toutes usines : jour le plus chargé {total_jour['Avant'].max()} ({total_jour['Avant'].idxmax():%d/%m}) → "
      f"{total_jour['Après'].max()} ({total_jour['Après'].idxmax():%d/%m})")

#### Analyse : camions reçus par jour et par usine

**Toutes les usines reçoivent beaucoup moins de camions chaque jour.** En moyenne, PLT_001 passe de 18,6 à 7,0 camions par jour, PLT_002 de 19,7 à 9,5, PLT_003 de 6,9 à 3,8 et PLT_004 de 18,5 à 6,9.

**Les pics disparaissent.** Le cas le plus net est PLT_002.  Ses pics des 26 et 27/11 (53 et 54 camions, repérés dès le milestone 1) tombent à 13 camions, et son jour le plus chargé passe de 54 à 23 camions. Toutes usines confondues, le jour le plus chargé passe de 118 camions (le 26/11) à 67.

**Certains jours reçoivent plus de camions qu'avant**, par exemple PLT_002 autour du 20/11 (12 camions au lieu de 3). C'est l'effet du lissage avec des jours fixes de livraison qui vont sur les jours creux pour équilibrer la semaine.

**Le 09/11 reste un pic, et c'est attendu.** C'est le premier jour du planning, les besoins du début de l'horizon ne peuvent pas être livrés en avance, donc ils partent le jour même (camions de démarrage, voir le milestone 3). C'est pourquoi PLT_001 et PLT_004 reçoivent ce jour-là 18 et 19 camions. Une fois le planning en régime, PLT_004 ne dépasse jamais 11 camions par jour. En exploitation réelle, ce pic n'existerait pas, car le planning de la période précédente aurait déjà anticipé ces besoins.

#### Jours de livraison des couples les plus actifs

Même carte de chaleur qu'au milestone 2 : les 10 couples qui livrent le plus souvent dans la baseline, avec à droite les mêmes couples après consolidation. Une livraison correspond à un couple et un jour, plusieurs camions le même jour comptent pour une seule livraison. Chaque case donne le nombre de livraisons ce jour de la semaine, sur les 6 semaines du planning (6 au maximum).
Il est important d'effectuer cette comparaison car c'etait les couples avec un fort potentiel, on va donc voir si nous avons su les optimiser.

In [ ]:
#  Jours de livraison des 10 couples les plus actifs 
JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi"]
camions["couple"] = camions["supplier_id"] + " × " + camions["plant"]

# Une livraison = un couple, un jour (plusieurs camions le même jour comptent pour une livraison)
livraisons = camions.drop_duplicates(["version", "couple", "date"])
tables = {v: pd.crosstab(livraisons.loc[livraisons["version"] == v, "couple"],
                         livraisons.loc[livraisons["version"] == v, "jour"]).reindex(columns=range(5), fill_value=0)
          for v in ["Avant", "Après"]}
top_10 = tables["Avant"].sum(axis=1).nlargest(10).index      # mêmes couples qu'au milestone 2
nb_semaines = camions["semaine"].nunique()

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), sharey=True)
for ax, version in zip(axes, ["Avant", "Après"]):
    heat = tables[version].reindex(top_10, fill_value=0)
    im = ax.imshow(heat.values, cmap="YlGnBu", vmin=0, vmax=nb_semaines, aspect="auto")
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            v = heat.values[i, j]
            ax.text(j, i, v if v else "", ha="center", va="center", fontsize=9,
                    color="white" if v > nb_semaines / 2 else "black")
    ax.set_xticks(range(5), JOURS)
    ax.set_yticks(range(len(top_10)), top_10)
    titre = "Avant : baseline" if version == "Avant" else "Après : planning consolidé"
    ax.set_title(f"{titre}\n{int(heat.values.sum())} livraisons, soit {nombre(heat.values.sum() / nb_semaines / 10)} par couple et par semaine")
fig.colorbar(im, ax=axes, label=f"Nombre de livraisons sur les {nb_semaines} semaines", shrink=0.8)
fig.suptitle("Jours de livraison des 10 couples les plus actifs de la baseline", fontsize=14)
plt.show()

#  Chiffres pour l'analyse
par_sem = livraisons.groupby(["version", "couple", "semaine"])["date"].nunique()   # jours de livraison par semaine
moy = par_sem.groupby(["version", "couple"]).mean()
for v in ["Avant", "Après"]:
    print(f"{v} : {moy[v].mean():.2f} jours de livraison par semaine en moyenne (couples livrés), "
          f"{(moy[v] > 2).sum()} couples à plus de 2 jours par semaine sur {len(moy[v])}")

# Les couples qui livrent encore plus de 2 jours par semaine après : camions optimisés / camions réservés
apres_c = camions[camions["version"] == "Après"]
for couple in moy["Après"][moy["Après"] > 2].index:
    nb = apres_c[apres_c["couple"] == couple].groupby("reserve").size()
    print(f"  {couple} : {nb.get(False, 0)} camions optimisés, {nb.get(True, 0)} camions réservés")

#### Analyse : jours de livraison des couples les plus actifs

**Avant, ces couples livraient tous les jours.** Les 10 couples les plus actifs de la baseline livrent les 5 jours de la semaine, presque chaque semaine : 291 livraisons sur les 6 semaines, soit 4,8 par couple et par semaine. 8 d'entre eux livrent PLT_004.

**Après, chaque couple a son jour fixe, et une livraison par semaine lui suffit.** Ces mêmes couples ne font plus que 60 livraisons, une par semaine. La case foncée de chaque ligne est le jour fixe du couple : par exemple le lundi pour SUP_00085 -> PLT_004, ou le jeudi pour SUP_00011 -> PLT_004. Les cases à "1" le lundi 09/11 ou le mardi 10/11 sont les camions de démarrage de la première semaine.

**Un couple à 2 jours fixes n'utilise souvent qu'un des deux jours.** Par exemple, SUP_00091 ->  PLT_004 a pour profil "mercredi + vendredi"(sortie 1), mais un seul camion par semaine lui suffit presque toujours : la fenêtre de 7 jours couvre toute la semaine. Le deuxième jour sert de réserve quand les besoins dépassent la capacité d'un camion.

**Sur l'ensemble des couples**, le nombre moyen de jours de livraison par semaine passe de 2,0 à 1,1. Seuls 4 couples sur 180 livrent encore plus de 2 jours par semaine, contre 52 avant. Ces 4 couples ont surtout des lignes sans poids (par exemple 24 camions réservés sur 29 pour SUP_00129 -> PLT_003) : leurs camions réservés partent le jour du besoin, en dehors des jours fixes. Pour eux, seule une meilleure qualité des données permettrait de regrouper les livraisons.

--- 

### Étude des économies réalisées (réalisé vs potentiel)

#### Les 10 couples les plus chers

Même classement qu'au milestone 2 : les 10 couples fournisseur × usine qui coûtent le plus cher dans la baseline, camions réservés compris. Pour chacun, on compare le coût avant et après consolidation. La partie hachurée est le coût des camions réservés (lignes sans poids) : il est identique avant et après, car ces lignes ne peuvent pas être consolidées.

In [ ]:
#  Coût des 10 couples les plus chers de la baseline : avant / après 
from matplotlib.patches import Patch

# Coût total de chaque couple, camions réservés compris (comme au milestone 2), et la part des camions réservés
cout_couple = camions.pivot_table(index="couple", columns="version", values="cout", aggfunc="sum", fill_value=0)[["Avant", "Après"]]
cout_reserve = (camions[camions["reserve"]]
                .pivot_table(index="couple", columns="version", values="cout", aggfunc="sum", fill_value=0)
                .reindex(index=cout_couple.index, columns=["Avant", "Après"], fill_value=0))
top_cher = cout_couple.nlargest(10, "Avant").index[::-1]          # du moins cher au plus cher (le plus cher en haut)

fig, ax = plt.subplots(figsize=(11, 6.5))
y = np.arange(len(top_cher))
h = 0.4
for decalage, version in [(h / 2, "Avant"), (-h / 2, "Après")]:
    total = cout_couple.loc[top_cher, version]
    reserve = cout_reserve.loc[top_cher, version]
    ax.barh(y + decalage, total - reserve, height=h, color=COULEURS[version], edgecolor="black",
            label="Avant : baseline" if version == "Avant" else "Après : planning consolidé")
    ax.barh(y + decalage, reserve, left=total - reserve, height=h, color=COULEURS[version], edgecolor="black",
            hatch="///", alpha=0.6)
    for yi, v in zip(y + decalage, total):
        ax.text(v, yi, f" {eur(v)}", va="center", fontsize=8)
ax.set_yticks(y, top_cher)
ax.set_xlabel("Coût total sur les 6 semaines (€)")
ax.xaxis.set_major_formatter(lambda v, _: nombre(v, 0))
avant_top, apres_top = cout_couple.loc[top_cher, "Avant"].sum(), cout_couple.loc[top_cher, "Après"].sum()
ax.set_title(f"Coût des 10 couples les plus chers de la baseline (camions réservés compris)\n"
             f"{eur(avant_top)} → {eur(apres_top)} ({pourcent(apres_top / avant_top - 1)})")
poignees, _ = ax.get_legend_handles_labels()
ax.legend(poignees + [Patch(facecolor="white", edgecolor="black", hatch="///")],
          ["Avant : baseline", "Après : planning consolidé", "dont camions réservés (poids inconnu)"], loc="lower right")
ax.grid(axis="x", alpha=0.5)
ax.set_axisbelow(True)
ax.margins(x=0.12)
plt.tight_layout()
plt.show()

#  Chiffres pour l'analyse 
detail_top = pd.DataFrame({"avant": cout_couple.loc[top_cher, "Avant"], "après": cout_couple.loc[top_cher, "Après"],
                           "dont réservés": cout_reserve.loc[top_cher, "Avant"]})
detail_top["économie"] = detail_top["avant"] - detail_top["après"]
detail_top["économie (%)"] = (detail_top["économie"] / detail_top["avant"] * 100).round(1)
print(detail_top[::-1].round(0))
reserve_top = detail_top["dont réservés"].sum()
print(f"Hors camions réservés : {eur(avant_top - reserve_top)} → {eur(apres_top - reserve_top)} "
      f"({pourcent((apres_top - reserve_top) / (avant_top - reserve_top) - 1)})")
print(f"Part des 10 couples dans le coût de la baseline : {pourcent(avant_top / cout_couple['Avant'].sum())} ; "
      f"part dans l'économie totale : {pourcent((avant_top - apres_top) / (cout_couple['Avant'].sum() - cout_couple['Après'].sum()))}")

#### Analyse : les 10 couples les plus chers

**Les couples aux données complètes baissent fortement.** SUP_00085 -> PLT_004, le couple le plus cher aux données complètes, passe de 7 429 € à 1 988 € (−73 %). SUP_00146 -> PLT_002 (−72 %), SUP_00144 -> PLT_001 (−63 %), SUP_00059 -> PLT_003 (−59 %) et SUP_00118 -> PLT_001 (−56 %) suivent la même tendance. Ce sont tous des couples qui livraient 3 à 5 jours par semaine dans la baseline : les regrouper sur 1 ou 2 jours fixes est ce qui rapporte le plus.

**Les couples aux poids manquants ne baissent presque pas.** Pour SUP_00153 -> PLT_004 et SUP_00001 -> PLT_004, l'économie est nulle : leur coût vient entièrement, ou presque, des camions réservés (lignes sans poids), qu'on ne peut donc pas optimiser. SUP_00129 ne gagne que 16 % vers PLT_003 et 3 % vers PLT_001, pour la même raison. Nous l'avions dit dans le milestone 2, pour ces couples, l'économie passe d'abord par la qualité des données.

**Au total, ces 10 couples passent de 48 749 € à 30 808 € (−36,8 %).** Hors camions réservés, la baisse atteint **−61,1 %** (de 29 343 € à 11 401 €), bien plus que la moyenne. Ces 10 couples pèsent 27,5 % du coût de la baseline et apportent 27,5 % de l'économie totale.

#### Zoom sur une semaine : l'exemple du cours

On reprend l'exemple du document du cours (semaine du 16/11, 5 couples très actifs), cette fois avec notre planning : pour chaque couple, le coût actuel (baseline) et le coût après notre optimisation. Les camions réservés sont exclus, car ils sont identiques avant et après.

In [ ]:
# coût de 5 couples sur la semaine du 16/11, avant / après 
SEMAINE = 47   # semaine du lundi 16/11
COUPLES_DIAPO = ["SUP_00085 × PLT_004", "SUP_00144 × PLT_001", "SUP_00059 × PLT_003",
                 "SUP_00080 × PLT_004", "SUP_00149 × PLT_004"]          # les 5 couples de l'exemple du cours

# Camions de la semaine, hors camions réservés (identiques avant et après)
sem = camions[~camions["reserve"] & (camions["semaine"] == SEMAINE) & camions["couple"].isin(COUPLES_DIAPO)]
diapo = sem.pivot_table(index="couple", columns="version", values="cout", aggfunc="sum").reindex(COUPLES_DIAPO)
nb_camions = sem.pivot_table(index="couple", columns="version", values="camion_id", aggfunc="size").reindex(COUPLES_DIAPO)

total_av, total_ap = diapo["Avant"].sum(), diapo["Après"].sum()
c85 = diapo.loc["SUP_00085 × PLT_004"]

fig = plt.figure(figsize=(15, 6))
grille = fig.add_gridspec(3, 2, width_ratios=[1.9, 1], hspace=0.35, wspace=0.08)
ax = fig.add_subplot(grille[:, 0])

y = np.arange(len(COUPLES_DIAPO))[::-1]          # le premier couple en haut
h = 0.38
for decalage, version, nom in [(h / 2, "Avant", "Coût actuel (€ / semaine)"), (-h / 2, "Après", "Coût après consolidation")]:
    barres = ax.barh(y + decalage, diapo[version], height=h, color=COULEURS[version], label=nom)
    ax.bar_label(barres, labels=[nombre(v, 0) for v in diapo[version]], padding=4, fontsize=11)
ax.set_yticks(y, COUPLES_DIAPO, fontsize=11)
ax.set_title("Coût de transport, semaine du 16/11 (€)", fontsize=15, fontweight="bold")
ax.xaxis.set_major_formatter(lambda v, _: nombre(v, 0))
ax.grid(axis="x", alpha=0.4)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.margins(x=0.1)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False, fontsize=10)

#  Les 3 cartes chiffrées 
cartes = [
    (f"{pourcent(total_ap / total_av - 1, 0)}",
     f"coût des 5 couples : {eur(total_av)} → {eur(total_ap)} / semaine", "#e0a84f"),
    (eur(total_av - total_ap), "économisés sur une semaine pour 5 couples", "#e8eff1"),
    (eur(c85["Avant"] - c85["Après"]),
     f"SUP_00085 × PLT_004 : {int(nb_camions.loc['SUP_00085 × PLT_004', 'Avant'])} camions ({nombre(c85['Avant'], 0)} €)\n→ "
     f"{int(nb_camions.loc['SUP_00085 × PLT_004', 'Après'])} camion le lundi ({nombre(c85['Après'], 0)} €)", "#e8eff1"),
]
for i, (chiffre, texte, couleur) in enumerate(cartes):
    carte = fig.add_subplot(grille[i, 1])
    carte.set_facecolor(couleur)
    carte.set_xticks([]), carte.set_yticks([])
    for bord in carte.spines.values():
        bord.set_visible(False)
    carte.text(0.06, 0.62, chiffre, fontsize=26, fontweight="bold", color="#1f3b4d", transform=carte.transAxes)
    carte.text(0.06, 0.15, texte, fontsize=10.5, color="#1f3b4d", transform=carte.transAxes, wrap=True)

fig.text(0.66, 0.02, "Camions réservés exclus (identiques avant et après).", fontsize=9, color="dimgray")
plt.show()

#  Chiffres pour l'analyse 
diapo["camions avant"] = nb_camions["Avant"]
diapo["camions après"] = nb_camions["Après"]
print(diapo.round(2).to_string())
print(f"Total : {eur(total_av)} → {eur(total_ap)} ({pourcent(total_ap / total_av - 1)}), économie {eur(total_av - total_ap)}")
print("Jour du camion après :", sem[sem["version"] == "Après"].groupby("couple")["date"].first().dt.strftime("%d/%m").to_dict())

#### Analyse : la semaine du 16/11
v 
**L'hypothèse tarifaire du milestone 2 est confirmée.** On avait appliqué le tarif d'un seul envoi au-delà de 4 000 kg, « à confirmer ». Dans le planning réel, les 5 camions pèsent tous entre 5,4 t et 16 t : ils sont bien chiffrés dans la dernière tranche de la grille.

**SUP_00085 → PLT_004 est l'exemple le plus parlant.** Il envoyait 5 camions dans la semaine pour 1 215 €. Il n'en envoie plus qu'un, le lundi 16/11, avec les mêmes 11,7 t, pour 331 € : **884 € d'économie sur une seule semaine**.

**Ces 5 couples sont les cas les plus favorables** : ils livrent beaucoup, tous les jours, et ont des données complètes. C'est pourquoi leur économie (−68 %) est bien supérieure à celle de l'ensemble du planning (−37 %), où beaucoup de couples livraient déjà peu souvent ou ont des lignes sans poids.

#### Part de chaque usine dans l'économie

Le milestone 2 avait chiffré le coût de chaque usine dans la baseline. On le compare ici au coût après consolidation, camions réservés compris (ils sont identiques des deux côtés), pour voir quelle usine contribue le plus à l'économie.

In [ ]:
#  Part de chaque usine dans l'économie 
# Coût de chaque usine avant / après, camions réservés compris (ils sont identiques des deux côtés)
usines = camions.pivot_table(index="plant", columns="version", values="cout", aggfunc="sum")[["Avant", "Après"]]
usines["Économie"] = usines["Avant"] - usines["Après"]
usines["Économie (%)"] = usines["Économie"] / usines["Avant"]
usines["Part du coût de la baseline"] = usines["Avant"] / usines["Avant"].sum()
usines["Part de l'économie totale"] = usines["Économie"] / usines["Économie"].sum()
usines["Camions réservés"] = camions[camions["reserve"] & (camions["version"] == "Avant")].groupby("plant")["cout"].sum()
usines["Part des réservés dans la baseline"] = usines["Camions réservés"] / usines["Avant"]
usines["Économie hors réservés (%)"] = 1 - (usines["Après"] - usines["Camions réservés"]) / (usines["Avant"] - usines["Camions réservés"])

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(usines))
l = 0.38
for decalage, version in [(-l / 2, "Avant"), (l / 2, "Après")]:
    barres = ax.bar(x + decalage, usines[version], width=l, color=COULEURS[version], edgecolor="black",
                    label="Avant : baseline" if version == "Avant" else "Après : planning consolidé")
    ax.bar_label(barres, labels=[eur(v) for v in usines[version]], fontsize=9, padding=2)
for xi, (eco, pct) in enumerate(zip(usines["Économie"], usines["Économie (%)"])):   # économie au-dessus de chaque usine
    ax.annotate(f"−{eur(eco)} (−{pourcent(pct, 0)})", (xi, usines["Avant"].max() * 1.12), ha="center",
                fontsize=10, fontweight="bold", color="#2e7d32")
ax.set_xticks(x, usines.index)
ax.set_ylabel("Coût total sur les 6 semaines (€)")
ax.yaxis.set_major_formatter(lambda v, _: nombre(v, 0))
ax.set_ylim(0, usines["Avant"].max() * 1.22)
ax.set_title(f"Coût de transport par usine (camions réservés compris)\n"
             f"économie totale : {eur(usines['Économie'].sum())} ({pourcent(usines['Économie'].sum() / usines['Avant'].sum())})")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
ax.grid(axis="y", alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

#  Chiffres pour l'analyse 
tableau = usines.copy()
for col in ["Avant", "Après", "Économie", "Camions réservés"]:
    tableau[col] = tableau[col].map(eur)
for col in ["Économie (%)", "Part du coût de la baseline", "Part de l'économie totale",
            "Part des réservés dans la baseline", "Économie hors réservés (%)"]:
    tableau[col] = tableau[col].map(pourcent)
tableau

#### Analyse : part de chaque usine dans l'économie

**PLT_004 apporte la plus grosse économie : 24 072 €, soit 37 % de l'économie totale** (−41 %). C'est ce qu'on supposait dans le milestone 2 : c'est l'usine la plus chère, sa charge est stable et ses fournisseurs livraient tous les jours (8 des 10 couples les plus actifs la livrent). Les regrouper sur un jour fixe est ce qui rapporte le plus.

**PLT_003 économise le moins en proportion (−29 %)**, et contribue moins à l'économie (12,5 %) qu'au coût de la baseline (15,8 %). La raison est la qualité des données : les camions réservés (lignes sans poids) représentent près de 30 % de son coût, une part que le moteur ne peut pas réduire. PLT_002 est dans le même cas, dans une moindre mesure (25 % de camions réservés, −34 %).

**Hors camions réservés, les 4 usines économisent presque autant** : entre −42 % (PLT_003) et −47 % (PLT_004). Les écarts entre usines viennent donc surtout des données manquantes, et non de la consolidation elle-même : le moteur est aussi efficace dans toutes les usines.

--- 

## Conclusion

## Conclusion

**La solution est fiable.** à faire 

**La consolidation change profondément la façon de livrer.**
- **Moins de camions, mieux remplis** : on passe de 1 741 à 736 camions (÷ 2,4). Le poids médian d'un camion passe de 323 kg à 806 kg, et la part des camions au-delà de 4 000 kg (la dernière tranche de la grille, où le prix est fixe) de 4,4 % à 16,3 %. Ces camions transportent désormais 70 % du poids.
- **Moins cher à la tonne** : le coût moyen d'une tonne transportée passe de 91 € à 50 €. Les petits camions de moins de 500 kg, qui coûtaient 39 % du budget pour 13 % du poids, n'en coûtent plus que 17 %.
- **Des livraisons régulières** : les couples les plus actifs passaient de 5 livraisons par semaine à une seule, le jour fixe de leur profil. En moyenne, un couple livre 1,1 jour par semaine au lieu de 2,0.
- **Des usines moins chargées** : les pics de réception disparaissent. Le jour le plus chargé de PLT_002 passe de 54 à 23 camions.

**L'économie est importante et proche du maximum possible.**
- Le coût total passe de **177 321 € à 112 195 € sur les 6 semaines, soit 65 126 € économisés (−36,7 %)**.
- C'est **96 % du potentiel** : sans aucune contrainte de jours, la consolidation économiserait 67 807 €. Les jours fixes et le lissage, demandés par le métier, ne coûtent que 2 682 €.
- L'exemple du cours est retrouvé sur nos données : sur la semaine du 16/11, les 5 couples de l'exemple passent de 3 994 € à 1 263 € (−68 %).
- Toutes les usines y contribuent. PLT_004 apporte la plus grosse part (24 072 €, 37 % de l'économie).

**Le principal gisement restant est la qualité des données.** Les 134 lignes sans poids partent chacune dans un camion réservé, qui ne peut pas être consolidé : elles coûtent 33 506 € dans toutes les versions, soit près de 30 % du planning final. Ce sont elles qui expliquent les couples et les usines qui économisent le moins (SUP_00129, SUP_00153, PLT_003). Hors camions réservés, chaque usine économise entre 42 % et 47 %. Renseigner ces poids est le levier le plus efficace pour aller plus loin, devant toute amélioration de l'algorithme.

**Limites.** Le remplissage reste faible dans l'absolu (3,3 % en médiane), car la plupart des couples ont moins de 4 t de besoins par semaine. Pour remplir réellement les camions, il faudrait sortir du périmètre du projet : mélanger plusieurs fournisseurs dans un même camion (tournées), ou livrer plus de 7 jours en avance. Enfin, la ligne de plus de 24,5 t (SUP_00074, 26,9 t) reste exclue en attendant la réponse du client.